# Conventional baseline

Inputs: prepared data tapes and retained Kaggle logs. Executes two real small-model updates through revllm.train, then checks the full-run evidence. The demonstration is not a 50M-token assignment run.

In [1]:
from pathlib import Path
import json, csv, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src' / 'revllm').exists(), 'Execute with repository root as working directory'
sys.path.insert(0, str(ROOT / 'src'))
from revllm.data import digest_file
def read_final(path):
    result = json.loads(path.read_text())
    updates = [json.loads(line) for line in (path.parent/'metrics.jsonl').read_text().splitlines()]
    assert sum(row['valid_targets'] for row in updates if row['status']=='COMMITTED') == result['committed_targets']
    return result


In [2]:
# Real two-update execution; excluded from assignment token totals.
from tempfile import TemporaryDirectory
from revllm.model import ModelConfig
from revllm.train import RunConfig, train
with TemporaryDirectory(prefix='revllm-notebook-') as scratch:
    cfg = RunConfig(run_id='notebook-demo',
                    model=ModelConfig(n_layer=2, n_embd=16, n_head=2, block_size=16,
                                      integrator='conventional', backward_mode='stored', step_size=0.5),
                    data_dir=ROOT/'data', out_dir=Path(scratch), physical_batch=2, effective_batch=2,
                    target_tokens=256, warmup_tokens=128, use_amp=False)
    measured = train(cfg, max_updates=2)
    assert measured['committed_targets'] == 64
    print({'demonstration': 'conventional', 'successful_updates': measured['updates'],
           'committed_targets': measured['committed_targets'], 'assignment_run': False})


{'demonstration': 'conventional', 'successful_updates': 2, 'committed_targets': 64, 'assignment_run': False}


In [3]:
path = ROOT/'experiments'/'kaggle-pilots-v3'/'runs'/'pilot-conventional'
final = json.loads((path/'final.json').read_text())
updates = [json.loads(x) for x in (path/'metrics.jsonl').read_text().splitlines()]
print({'run_id': final['run_id'], 'committed_targets': final['committed_targets'], 'updates': final['updates'], 'dev_loss_nats': json.loads((path/'eval.jsonl').read_text().splitlines()[-1])['dev_loss_nats'], 'raw_update_records': len(updates)})

{'run_id': 'pilot-conventional', 'committed_targets': 2000000, 'updates': 977, 'dev_loss_nats': 3.9543055798858404, 'raw_update_records': 977}


In [4]:
for campaign in ('kaggle-full-v1','kaggle-locked-v1','kaggle-locked-repair-v1'):
    for path in sorted((ROOT/'experiments'/campaign/'runs').glob('A-*/final.json')):
        r=read_final(path)
        print({k:r[k] for k in ('run_id','committed_targets','updates','holdout_loss_nats','training_tokens_per_second')})

{'run_id': 'A-1337', 'committed_targets': 50000000, 'updates': 888, 'holdout_loss_nats': 2.851997889526367, 'training_tokens_per_second': 81771.96577904782}
{'run_id': 'A-2027', 'committed_targets': 50000000, 'updates': 888, 'holdout_loss_nats': 2.8502183413085938, 'training_tokens_per_second': 83802.45686496505}
{'run_id': 'A-L-1337', 'committed_targets': 50000000, 'updates': 888, 'holdout_loss_nats': 2.854566508911133, 'training_tokens_per_second': 76890.6070689729}
{'run_id': 'A-L-2027', 'committed_targets': 50000000, 'updates': 888, 'holdout_loss_nats': 2.853359813232422, 'training_tokens_per_second': 81847.60991968475}
